# gradient-routing-slm: full run on a Kaggle T4
Settings: **Accelerator = GPU T4** (x1 or x2, only one is used), **Internet = On**. Then Run All.
Results land in `/kaggle/working/out/` (plus `run.log`).
Set `ROUTED = True` in the first cell to train the gradient-routed model instead (outputs in `out_routed/`).

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/gradient-routing-slm.git"  # <-- set this
ROUTED = False   # True: train with gradient routing (configs/routed.py); results in /kaggle/working/out_routed

In [ ]:
!pip install -q -U tiktoken datasets

In [ ]:
!rm -rf /tmp/repo && git clone --depth 1 {REPO_URL} /tmp/repo
%cd /tmp/repo
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess, shutil, os, json
rc = None
OUT = "out_routed" if ROUTED else "out"
env = {**os.environ, "ROUTED": "1" if ROUTED else "0"}
try:
    p = subprocess.Popen(["bash", "run_all.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    with open("run.log", "w") as log:
        for line in p.stdout:
            print(line, end="")
            log.write(line)
    rc = p.wait()
finally:
    os.makedirs(f"/kaggle/working/{OUT}", exist_ok=True)
    if os.path.isdir(OUT):
        shutil.copytree(OUT, f"/kaggle/working/{OUT}", dirs_exist_ok=True)
    if os.path.exists("run.log"):
        shutil.copy("run.log", "/kaggle/working/run.log")
    if os.path.exists("data/out/meta.json"):
        shutil.copy("data/out/meta.json", f"/kaggle/working/{OUT}/data_meta.json")
assert rc == 0, f"run_all.sh failed with exit code {rc} - see run.log"
mode = json.load(open("data/out/meta.json"))["mode"]
print()
print("DATA MODE:", mode, "" if mode == "real" else " <-- WARNING: HF download failed, results are on synthetic data!")
print(os.listdir(f"/kaggle/working/{OUT}"))

In [ ]:
from IPython.display import Image, display
display(Image(f"/kaggle/working/{OUT}/loss_curves.png"))
display(Image(f"/kaggle/working/{OUT}/ablation_heatmap.png"))
print(open(f"/kaggle/working/{OUT}/ablation_summary.txt").read())
print(open(f"/kaggle/working/{OUT}/samples.txt").read())
if ROUTED:
    print(open(f"/kaggle/working/{OUT}/routing_stats.csv").read().splitlines()[-1])